# Cows vs Horses 
## CycleGAN Unpaired Style Transfer


## 1.1 Imports & Device

In [ ]:
import os, shutil, itertools
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from torchvision import datasets, transforms
from torchvision.utils import make_grid, save_image
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1.2 Paths & Constants

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
TRAIN_PATH  = 'data/Newdata/Train'
TEST_PATH   = 'data/Newdata/Test'

GAN_OUT     = 'data/generated'      # GAN synthetic images
MERGED_PATH = 'data/new_train'      # Real + GAN merged (Step 2)
NST_OUT     = 'data/nst_results'    # CycleGAN outputs (Step 3)

# ── Hyper-parameters ───────────────────────────────────────────────────────
IMAGE_SIZE    = 128
LATENT_DIM    = 100
NUM_CLASSES   = 2
GAN_EPOCHS    = 1000
GAN_BATCH     = 8
IMAGES_TO_GEN = 80      # synthetic images per class
CNN_EPOCHS    = 40
CNN_BATCH     = 12
CYCLEGAN_EPOCHS = 160  # Step 3

CLASSES = ['Cow', 'Horse']   # must match folder names exactly

## 1.3 Transforms

In [ ]:
# ── GAN transform: no augmentation so GAN learns the real distribution ───
gan_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# ── CNN train transform: augmentation to fight overfitting ───────────────
cnn_train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# ── CNN test transform: no augmentation ────────────────────────────────
cnn_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

## 1.4 CNN Architecture

In [ ]:
class ConvolutionalNetwork(nn.Module):
    """
    4-block CNN with BatchNorm, Dropout and Global Average Pooling.
    Input : (B, 3, 128, 128)
    Output: (B, 2) logits
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   32,  3, padding=1);  self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32,  64,  3, padding=1);  self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64,  128, 3, padding=1);  self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1);  self.bn4 = nn.BatchNorm2d(256)
        self.pool    = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)
        self.gap     = nn.AdaptiveAvgPool2d(1)
        self.fc1     = nn.Linear(256, 128)
        self.fc2     = nn.Linear(128, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.gap(x).view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

## 1.5 Load Baseline Dataset (Real Images Only)

In [ ]:
TRAIN_PATH_3  = 'data/Newdata3/Train'

baseline_train_data = datasets.ImageFolder(root=TRAIN_PATH_3, transform=cnn_train_transform)
baseline_test_data  = datasets.ImageFolder(root=TEST_PATH,  transform=cnn_test_transform)

baseline_train_loader = DataLoader(baseline_train_data, batch_size=CNN_BATCH, shuffle=True)
baseline_test_loader  = DataLoader(baseline_test_data,  batch_size=CNN_BATCH, shuffle=False)

print(f'Baseline training samples : {len(baseline_train_data)}')
print(f'Baseline test samples     : {len(baseline_test_data)}')
print(f'Class mapping             : {baseline_train_data.class_to_idx}')

# Visualise a few real training images
images, labels = next(iter(baseline_train_loader))
grid = make_grid(images[:8], nrow=4, normalize=True)
plt.figure(figsize=(10, 4))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title('Sample Real Training Images')
plt.axis('off')
plt.show()

## 1.6 Train Baseline CNN

In [ ]:
def train_cnn(model, train_loader, test_loader, train_data, test_data,
              epochs=CNN_EPOCHS, label='CNN'):
    """Generic CNN training loop. Returns (train_losses, test_losses, train_accs, test_accs)."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    tr_losses, ts_losses, tr_accs, ts_accs = [], [], [], []

    for epoch in range(1, epochs + 1):
        model.train()
        trn_corr, trn_loss = 0, 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out  = model(xb)
            loss = criterion(out, yb)
            loss.backward(); optimizer.step()
            trn_corr += (out.argmax(1) == yb).sum().item()
            trn_loss += loss.item()

        avg_tr_loss = trn_loss / len(train_loader)
        tr_acc      = trn_corr / len(train_data) * 100

        model.eval()
        tst_corr, tst_loss = 0, 0.0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                tst_loss += criterion(out, yb).item()
                tst_corr += (out.argmax(1) == yb).sum().item()

        avg_ts_loss = tst_loss / len(test_loader)
        ts_acc      = tst_corr / len(test_data) * 100
        scheduler.step(avg_ts_loss)

        tr_losses.append(avg_tr_loss); ts_losses.append(avg_ts_loss)
        tr_accs.append(tr_acc);        ts_accs.append(ts_acc)

        print(f'[{label}] Epoch {epoch:02d}/{epochs} | '
              f'Train Loss: {avg_tr_loss:.4f} Acc: {tr_acc:.1f}% | '
              f'Test Loss: {avg_ts_loss:.4f} Acc: {ts_acc:.1f}%')

    print(f'\n🎯 [{label}] Final Test Accuracy: {ts_accs[-1]:.1f}%')
    return tr_losses, ts_losses, tr_accs, ts_accs


torch.manual_seed(42)
baseline_cnn = ConvolutionalNetwork().to(device)
print(f'CNN parameters: {sum(p.numel() for p in baseline_cnn.parameters() if p.requires_grad):,}')

bl_tr_losses, bl_ts_losses, bl_tr_accs, bl_ts_accs = train_cnn(
    baseline_cnn, baseline_train_loader, baseline_test_loader,
    baseline_train_data, baseline_test_data, label='Baseline CNN'
)

## 1.7 Baseline CNN Results & Confusion Matrix

In [ ]:
def plot_training(tr_losses, ts_losses, tr_accs, ts_accs, title='CNN'):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(tr_losses, label='Train'); axes[0].plot(ts_losses, label='Test')
    axes[0].set_title(f'{title} — Loss'); axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss'); axes[0].legend()
    axes[1].plot(tr_accs, label='Train'); axes[1].plot(ts_accs, label='Test')
    axes[1].set_title(f'{title} — Accuracy'); axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)'); axes[1].legend()
    plt.tight_layout(); plt.show()

def show_confusion_matrix(model, loader, title='Confusion Matrix'):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb.to(device)).argmax(1).cpu().numpy()
            all_preds.extend(preds); all_labels.extend(yb.numpy())
    cm   = confusion_matrix(all_labels, all_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, colorbar=False); ax.set_title(title)
    plt.tight_layout(); plt.show()
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels) * 100
    print(f'Accuracy: {acc:.1f}%')
    return acc

plot_training(bl_tr_losses, bl_ts_losses, bl_tr_accs, bl_ts_accs, 'Baseline CNN')
baseline_acc = show_confusion_matrix(baseline_cnn, baseline_test_loader, 'Baseline CNN — Confusion Matrix')

---
# Unpaired Neural Style Transfer with CycleGAN
CycleGAN learns to translate images between two unpaired domains
(Cow → Horse, Horse → Cow) **without any paired training data**.

## How CycleGAN Works
- **Two generators**: G: Cow→Horse, F: Horse→Cow
- **Two discriminators**: D_H (real/fake horse), D_C (real/fake cow)
- **Cycle consistency loss**: G(F(horse)) ≈ horse and F(G(cow)) ≈ cow
- **Identity loss**: G(horse) ≈ horse (preserves colour statistics)


## 3.1 CycleGAN Components — ResNet Block & Generator

In [ ]:
# ── Residual block used inside the CycleGAN generator ─────────────────
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels),
            nn.ReLU(True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3, bias=False),
            nn.InstanceNorm2d(channels)
        )
    def forward(self, x):
        return x + self.block(x)

# ── CycleGAN Generator (encoder–9 ResBlocks–decoder) ──────────────────
class CycleGenerator(nn.Module):
    """
    Translates images from one domain to another.
    Architecture: 3 downsampling → 9 ResBlocks → 3 upsampling
    Input/Output: (B, 3, IMAGE_SIZE, IMAGE_SIZE)
    """
    def __init__(self, n_res=9):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(3, 64, 7, bias=False), nn.InstanceNorm2d(64), nn.ReLU(True),
            # Downsampling
            nn.Conv2d(64,  128, 3, stride=2, padding=1, bias=False), nn.InstanceNorm2d(128), nn.ReLU(True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1, bias=False), nn.InstanceNorm2d(256), nn.ReLU(True),
        ]
        # Residual blocks
        layers += [ResBlock(256) for _ in range(n_res)]
        # Upsampling
        layers += [
            nn.ConvTranspose2d(256, 128, 3, stride=2, padding=1, output_padding=1, bias=False),
            nn.InstanceNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128,  64, 3, stride=2, padding=1, output_padding=1, bias=False),
            nn.InstanceNorm2d(64),  nn.ReLU(True),
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, 3, 7),
            nn.Tanh()
        ]
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

## 3.2 CycleGAN Discriminator (PatchGAN)

In [ ]:
class CycleDiscriminator(nn.Module):
    """
    PatchGAN discriminator: classifies overlapping 94 *94 patches as real/fake.
    Output: feature map (each value = real/fake for a local patch).
    """
    def __init__(self):
        super().__init__()
        def block(in_c, out_c, norm=True):
            layers = [nn.Conv2d(in_c, out_c, 4, stride=2, padding=1, bias=False)]
            if norm:
                layers.append(nn.InstanceNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, True))
            return layers

        self.model = nn.Sequential(
            *block(3,   64,  norm=False),
            *block(64,  128),
            *block(128, 256),
            *block(256, 512),
            nn.Conv2d(512, 1, 4, padding=1)
        )

    def forward(self, x):
        return self.model(x)

## 3.3 Image Buffer (Reduces Model Oscillation)

In [ ]:
class ImageBuffer:
    """
    Stores a buffer of 50 previously generated images.
    Returns a mix of old and new images to stabilise GAN training.
    """
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []

    def push_and_pop(self, images):
        result = []
        for img in images:
            img = img.unsqueeze(0)
            if len(self.data) < self.max_size:
                self.data.append(img)
                result.append(img)
            elif torch.rand(1).item() > 0.5:
                idx = torch.randint(len(self.data), (1,)).item()
                result.append(self.data[idx].clone())
                self.data[idx] = img
            else:
                result.append(img)
        return torch.cat(result, dim=0)

## 3.4 CycleGAN Dataset — Unpaired Cows & Horses

In [ ]:
from torch.utils.data import Dataset
from PIL import Image

class UnpairedDataset(Dataset):
    """
    Loads two unpaired image folders independently.
    Returns one random image from each domain per iteration.
    Domain A = Cow, Domain B = Horse
    """
    def __init__(self, root_a, root_b, transform=None):
        self.files_a = sorted([
            os.path.join(root_a, f) for f in os.listdir(root_a)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        self.files_b = sorted([
            os.path.join(root_b, f) for f in os.listdir(root_b)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        self.transform = transform

    def __len__(self):
        return max(len(self.files_a), len(self.files_b))

    def __getitem__(self, idx):
        img_a = Image.open(self.files_a[idx % len(self.files_a)]).convert('RGB')
        img_b = Image.open(self.files_b[idx % len(self.files_b)]).convert('RGB')
        if self.transform:
            img_a = self.transform(img_a)
            img_b = self.transform(img_b)
        return img_a, img_b

# CycleGAN uses InstanceNorm → needs images without norm=0.5
cycle_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))# will tran from [0,1] to [-1,1] bec tanh  use  tha 1,-1
])

cycle_dataset = UnpairedDataset(
    root_a=os.path.join(TRAIN_PATH, 'Cow'),
    root_b=os.path.join(TRAIN_PATH, 'Horse'),
    transform=cycle_transform
)
cycle_loader = DataLoader(cycle_dataset, batch_size=1, shuffle=True)

print(f'Cow images  (Domain A): {len(cycle_dataset.files_a)}')
print(f'Horse images (Domain B): {len(cycle_dataset.files_b)}')
print(f'Dataset length: {len(cycle_dataset)}')

## 3.5 Initialise CycleGAN Models & Optimisers

In [ ]:
torch.manual_seed(42)

# Two generators: G = Cow→Horse, F = Horse→Cow
G = CycleGenerator().to(device)   # Cow → Horse
F = CycleGenerator().to(device)   # Horse → Cow

# Two PatchGAN discriminators
D_H = CycleDiscriminator().to(device)  # judges horses
D_C = CycleDiscriminator().to(device)  # judges cows

# Losses
criterion_adv      = nn.MSELoss()   # LSGAN (more stable than BCE)
criterion_cycle    = nn.L1Loss()    # cycle-consistency
criterion_identity = nn.L1Loss()    # identity

# Optimisers — generators share one, discriminators share one
opt_G = optim.Adam(itertools.chain(G.parameters(), F.parameters()),
                   lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(itertools.chain(D_H.parameters(), D_C.parameters()),
                   lr=2e-4, betas=(0.5, 0.999))

# LR scheduler: constant for first half, linear decay for second half
def lr_lambda(epoch):
    half = CYCLEGAN_EPOCHS // 2
    return 1.0 if epoch < half else 1.0 - (epoch - half) / half

sched_G = optim.lr_scheduler.LambdaLR(opt_G, lr_lambda)
sched_D = optim.lr_scheduler.LambdaLR(opt_D, lr_lambda)

# Image buffers
buf_H = ImageBuffer()
buf_C = ImageBuffer()

# Weights
LAMBDA_CYCLE    = 10.0
LAMBDA_IDENTITY = 0.5

print(f'G (Cow→Horse) params : {sum(p.numel() for p in G.parameters() if p.requires_grad):,}')
print(f'F (Horse→Cow) params : {sum(p.numel() for p in F.parameters() if p.requires_grad):,}')
print(f'D_H params           : {sum(p.numel() for p in D_H.parameters() if p.requires_grad):,}')

## 3.6 CycleGAN Training Loop

In [ ]:

cow2horse_dir = os.path.join(NST_OUT, "cow2horse")
horse2cow_dir = os.path.join(NST_OUT, "horse2cow")
rec_cow_dir   = os.path.join(NST_OUT, "rec_cow")
rec_horse_dir = os.path.join(NST_OUT, "rec_horse")
from torchvision.utils import save_image

os.makedirs(cow2horse_dir, exist_ok=True)
os.makedirs(horse2cow_dir, exist_ok=True)
os.makedirs(rec_cow_dir, exist_ok=True)
os.makedirs(rec_horse_dir, exist_ok=True)

os.makedirs(NST_OUT, exist_ok=True)
cyc_g_losses, cyc_d_losses = [], []

# Pick 2 fixed images for visualisation
fixed_cow   = next(iter(DataLoader(cycle_dataset, batch_size=1)))[0].to(device)
fixed_horse = next(iter(DataLoader(cycle_dataset, batch_size=1)))[1].to(device)

for epoch in range(1, CYCLEGAN_EPOCHS + 1):
    epoch_g = epoch_d = 0.0
    G.train(); F.train(); D_H.train(); D_C.train()

    for real_cow, real_horse in cycle_loader:
        real_cow   = real_cow.to(device)
        real_horse = real_horse.to(device)
        B = real_cow.size(0)

        real_label = torch.ones(B, 1, *D_H(real_horse).shape[2:], device=device)
        fake_label = torch.zeros_like(real_label)

        # ── Generator update ─────────────────────────────────────────
        opt_G.zero_grad()

        # Identity loss (G should keep horses as horses, F cows as cows)
        id_horse  = G(real_horse)
        id_cow    = F(real_cow)
        loss_id   = (criterion_identity(id_horse, real_horse) +
                     criterion_identity(id_cow,   real_cow)) * LAMBDA_IDENTITY

        # Adversarial loss
        fake_horse = G(real_cow)
        fake_cow   = F(real_horse)
        loss_adv   = (criterion_adv(D_H(fake_horse), real_label) +
                      criterion_adv(D_C(fake_cow),   real_label))

        # Cycle-consistency loss: cow→horse→cow ≈ cow, horse→cow→horse ≈ horse
        rec_cow   = F(fake_horse)
        rec_horse = G(fake_cow)
        loss_cyc  = (criterion_cycle(rec_cow,   real_cow) +
                     criterion_cycle(rec_horse, real_horse)) * LAMBDA_CYCLE

        g_loss = loss_adv + loss_cyc + loss_id
        g_loss.backward(); opt_G.step()

        # ── Discriminator update ─────────────────────────────────────
        opt_D.zero_grad()

        # D_H: real horses vs fake (buffered) horses
        fake_horse_buf = buf_H.push_and_pop(fake_horse.detach())
        d_h_loss = 0.5 * (criterion_adv(D_H(real_horse),      real_label) +
                           criterion_adv(D_H(fake_horse_buf), fake_label))

        # D_C: real cows vs fake (buffered) cows
        fake_cow_buf = buf_C.push_and_pop(fake_cow.detach())
        d_c_loss = 0.5 * (criterion_adv(D_C(real_cow),       real_label) +
                           criterion_adv(D_C(fake_cow_buf),  fake_label))

        d_loss = d_h_loss + d_c_loss
        d_loss.backward(); opt_D.step()

        epoch_g += g_loss.item()
        epoch_d += d_loss.item()

    sched_G.step(); sched_D.step()
    cyc_g_losses.append(epoch_g); cyc_d_losses.append(epoch_d)

    if epoch % 20 == 0 or epoch == 1:
        G.eval(); F.eval()
        with torch.no_grad():
            cow_to_horse = G(fixed_cow).cpu()
            horse_to_cow = F(fixed_horse).cpu()
            rec_c        = F(G(fixed_cow)).cpu()
            rec_h        = G(F(fixed_horse)).cpu()

        grid = make_grid(
            torch.cat([fixed_cow.cpu(), cow_to_horse, rec_c,
                        fixed_horse.cpu(), horse_to_cow, rec_h], dim=0),
            nrow=3, normalize=True) 
        save_image(cow_to_horse, os.path.join(cow2horse_dir, f"epoch_{epoch}.png"), normalize=True)
        save_image(horse_to_cow, os.path.join(horse2cow_dir, f"epoch_{epoch}.png"), normalize=True)
        save_image(rec_c,        os.path.join(rec_cow_dir,   f"epoch_{epoch}.png"), normalize=True)
        save_image(rec_h,        os.path.join(rec_horse_dir, f"epoch_{epoch}.png"), normalize=True)
        
        plt.figure(figsize=(9, 6))
        plt.imshow(grid.permute(1, 2, 0).numpy())
        plt.title(f'Epoch {epoch} | Row1: Cow → Horse → RecCow | Row2: Horse → Cow → RecHorse')
        plt.axis('off'); plt.show()
        print(f'Epoch {epoch:>3}/{CYCLEGAN_EPOCHS} | G_loss: {epoch_g:.4f} | D_loss: {epoch_d:.4f}')

plt.figure(figsize=(8, 3))
plt.plot(cyc_g_losses, label='Generator (G+F)')
plt.plot(cyc_d_losses, label='Discriminator (D_H+D_C)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('CycleGAN Training Loss'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
print(cyc_g_losses)

In [ ]:
# Recursive search from current directory
for root, dirs, files in os.walk('.'):
    for file in files:
        if file.endswith('.pth'):
            print(os.path.join(root, file))

In [ ]:
import os

# Search common locations for .pth files
search_dirs = [
    '.',                    # current directory
    './checkpoints',
    './weights',
    './models',
    './saved_models',
    'data/',        # your home folder
]

for d in search_dirs:
    if os.path.exists(d):
        files = [f for f in os.listdir(d) if f.endswith('.pth')]
        if files:
            print(f"Found in {os.path.abspath(d)}:")
            for f in files:
                print(f"   {f}")

In [ ]:
print(G.eval())

## 3.7 CycleGAN Inference — Visualise Style Transfer Results

In [ ]:
G.eval(); F.eval()

# Sample 4 cows and 4 horses from the dataset
sample_loader = DataLoader(cycle_dataset, batch_size=4, shuffle=True)
sample_cows, sample_horses = next(iter(sample_loader))
sample_cows   = sample_cows.to(device)
sample_horses = sample_horses.to(device)

with torch.no_grad():
    fake_horses = G(sample_cows)    # Cow → Horse
    rec_cows    = F(fake_horses)    # Horse → Cow (cycle back)
    fake_cows   = F(sample_horses)  # Horse → Cow
    rec_horses  = G(fake_cows)      # Cow → Horse (cycle back)

def show_translation(originals, translated, reconstructed, title_orig, title_trans, title_rec):
    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    titles = [title_orig, title_trans, title_rec]
    for col in range(4):
        for row, imgs in enumerate([originals, translated, reconstructed]):
            img = imgs[col].cpu().permute(1, 2, 0).numpy()
            img = (img * 0.5 + 0.5).clip(0, 1)
            axes[row, col].imshow(img)
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_title(titles[row], fontsize=12, fontweight='bold')
    plt.suptitle('CycleGAN Unpaired Style Transfer', fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

print('=== Cow → Horse (and cycle back to Cow) ===')
show_translation(sample_cows, fake_horses, rec_cows,
                 'Real Cow', 'Translated Horse', 'Reconstructed Cow')

print('=== Horse → Cow (and cycle back to Horse) ===')
show_translation(sample_horses, fake_cows, rec_horses,
                 'Real Horse', 'Translated Cow', 'Reconstructed Horse')

## 3.8 Save CycleGAN Style-Transfer Outputs to Disk

In [ ]:
os.makedirs(os.path.join(NST_OUT, 'cow_to_horse'), exist_ok=True)
os.makedirs(os.path.join(NST_OUT, 'horse_to_cow'), exist_ok=True)

G.eval(); F.eval()
save_loader = DataLoader(cycle_dataset, batch_size=1, shuffle=False)
n_save = 500   # number of style-transferred images to save per direction

with torch.no_grad():
    for i, (cow, horse) in enumerate(save_loader):
        if i >= n_save:
            break
        fake_h = G(cow.to(device))
        fake_c = F(horse.to(device))
        save_image(fake_h, os.path.join(NST_OUT, 'cow_to_horse', f'nst_{i:04d}.png'), normalize=True)
        save_image(fake_c, os.path.join(NST_OUT, 'horse_to_cow', f'nst_{i:04d}.png'), normalize=True)

print(f'✅ Saved {n_save} Cow→Horse images  → {NST_OUT}/cow_to_horse/')
print(f'✅ Saved {n_save} Horse→Cow images  → {NST_OUT}/horse_to_cow/')

In [ ]:
cnn_train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
TRAIN_PATH_2  = 'data/Newdata2/Train'
TEST_PATH_2   = 'data/Newdata2/Test'


train_data = datasets.ImageFolder(root=TRAIN_PATH_2 , transform=cnn_train_transform)
test_data  = datasets.ImageFolder(root=TEST_PATH_2,   transform=cnn_test_transform)

train_loader = DataLoader(train_data, batch_size=CNN_BATCH, shuffle=True,  drop_last=False)
test_loader  = DataLoader(test_data,  batch_size=CNN_BATCH, shuffle=False)

print(f'Training samples : {len(train_data)}')
print(f'Test samples     : {len(test_data)}')
print(f'Class mapping    : {train_data.class_to_idx}')

# Show a merged batch
images, labels = next(iter(train_loader))
grid = make_grid(images[:12], nrow=4, normalize=True)
plt.figure(figsize=(12, 4))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title('Sample merged training images (Real + GAN)')
plt.axis('off')
plt.show()

In [ ]:
class ConvolutionalNetwork(nn.Module):
    """
    3-block CNN with BatchNorm, Dropout and Global Average Pooling.
    Input : (B, 3, 128, 128)
    Output: (B, 2) logits
    """
    def __init__(self):
        super().__init__()

        # ── Conv blocks ───────────────────────────────────────────────
        self.conv1 = nn.Conv2d(3,   32,  3, padding=1)
        self.bn1   = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32,  64,  3, padding=1)
        self.bn2   = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64,  128, 3, padding=1)
        self.bn3   = nn.BatchNorm2d(128)

        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn4   = nn.BatchNorm2d(256)

        self.pool    = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.4)

        # ── Global Average Pooling → collapses spatial dims ───────────
        self.gap = nn.AdaptiveAvgPool2d(1)

        # ── Fully connected ───────────────────────────────────────────
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128,  2)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x)

In [ ]:
import torch.nn.functional as F

In [ ]:
torch.manual_seed(42)
cnnmodel = ConvolutionalNetwork().to(device)

total_params = sum(p.numel() for p in cnnmodel.parameters() if p.requires_grad)
print(f'CNN trainable parameters: {total_params:,}')

criterion_cnn = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnnmodel.parameters(), lr=1e-3, weight_decay=1e-4)
cnn_scheduler = optim.lr_scheduler.ReduceLROnPlateau(cnn_optimizer, patience=5, factor=0.5)

train_losses, test_losses   = [], []
train_accs,   test_accs     = [], []

for epoch in range(1, CNN_EPOCHS + 1):

    # ── Train ──────────────────────────────────────────────────────────
    cnnmodel.train()
    trn_corr, trn_loss = 0, 0.0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        cnn_optimizer.zero_grad()
        y_pred = cnnmodel(x_batch)
        loss   = criterion_cnn(y_pred, y_batch)
        loss.backward()
        cnn_optimizer.step()

        trn_corr += (y_pred.argmax(1) == y_batch).sum().item()
        trn_loss += loss.item()

    avg_trn_loss = trn_loss / len(train_loader)
    trn_acc      = trn_corr / len(train_data) * 100

    # ── Evaluate ───────────────────────────────────────────────────────
    cnnmodel.eval()
    tst_corr, tst_loss = 0, 0.0

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            y_val   = cnnmodel(x_batch)
            tst_loss += criterion_cnn(y_val, y_batch).item()
            tst_corr += (y_val.argmax(1) == y_batch).sum().item()

    avg_tst_loss = tst_loss / len(test_loader)
    tst_acc      = tst_corr / len(test_data) * 100

    cnn_scheduler.step(avg_tst_loss)

    train_losses.append(avg_trn_loss)
    test_losses.append(avg_tst_loss)
    train_accs.append(trn_acc)
    test_accs.append(tst_acc)

    print(f'Epoch {epoch:02d}/{CNN_EPOCHS} | '
          f'Train Loss: {avg_trn_loss:.4f} | Train Acc: {trn_acc:.1f}% | '
          f'Test Loss: {avg_tst_loss:.4f} | Test Acc: {tst_acc:.1f}%')

print(f'\n🎯 Final Test Accuracy: {test_accs[-1]:.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train')
axes[0].plot(test_losses,  label='Test')
axes[0].set_title('CNN Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(train_accs, label='Train')
axes[1].plot(test_accs,  label='Test')
axes[1].set_title('CNN Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()

plt.tight_layout()
plt.show()